In [1]:
import tiktoken
import numpy as np
import requests
import pathlib
import tqdm

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
from datasets import load_dataset

# use name="sample-10BT" to use the 10BT sample
fw = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

In [6]:
token_counts_new = 0
MAX_TOKEN_COUNT = int(1e9)

with open("sample_text.txt","w") as f:
    for sample in fw:
        token_counts_new += sample["token_count"]
        if token_counts_new < MAX_TOKEN_COUNT:
            f.write(sample["text"]) 
        else:
            break

In [ ]:
CHUNK_SIZE = 100*1024*1024

def prepare_binary_data(read_file_path, data_chunk,tokenizer):
    with open(read_file_path,"r",encoding="utf-8") as fin:
        with open("train.bin","wb") as train_bin, open("test.bin","wb") as test_bin, open("valid.bin","wb") as valid_bin:
            i = 0
            while True:
                text = fin.read(data_chunk)

                if not text:
                    break

                tokens = tokenizer.encode(text)
                arr = np.asarray(tokens,dtype=np.uint16)
                if i < 400 :
                    arr.tofile(valid_bin)
                    i += 100
                elif i < 600:
                    arr.tofile(test_bin)
                    i += 100
                else:
                    arr.tofile(train_bin)





In [3]:
from pathlib import Path

Path("train.bin").resolve().parent/"data"/"train.bin"

PosixPath('/home/tuhin/python_codes/Tiny LLM/data/train.bin')

In [4]:
import os
for name in ["train.bin", "valid.bin", "test.bin"]:
    size = os.path.getsize(Path(name).resolve().parent/"data"/name)
    n_tokens = size // np.dtype(np.uint16).itemsize

    print(
        f"{name}: "
        f"{size / 1024**2:.2f} MiB, "
        f"{n_tokens:,} tokens"
    )

train.bin: 1628.95 MiB, 854,037,469 tokens
valid.bin: 180.77 MiB, 94,776,824 tokens
test.bin: 90.22 MiB, 47,301,015 tokens


In [5]:
import torch
from torch.utils.data import Dataset,DataLoader

In [6]:
class BinaryTextDataSet(Dataset):
    def __init__(self,path,context_length,stride):
        super().__init__()
        self.context_length = context_length
        self.stride = stride

        self.data = np.memmap(path,dtype = np.uint16,mode = "r")

        self.n_sample = (len(self.data) - context_length + stride - 1 )//stride

    def __len__(self):
        return self.n_sample

    def __getitem__(self, index):
        start = index * self.stride
        end = start + self.context_length

        input_id = torch.tensor(self.data[start:end].copy()).long()
        target_id = torch.tensor(self.data[start+1 : end+1].copy()).long()

        return input_id,target_id



In [7]:
filename = "valid"
path = Path(f"{filename}.bin").resolve().parent/"data"/f"{filename}.bin"
dataset = BinaryTextDataSet(path,1024,256)
len(dataset)

370218

In [8]:
dataloader = DataLoader(dataset,64,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)

In [9]:
input_tokens,output_tokens = next(iter(dataloader))
input_tokens.shape

/home/tuhin/miniconda3/envs/physics/lib/python3.12/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


torch.Size([64, 1024])

In [10]:
from Brain.transformer import Transformer

In [11]:
model = Transformer(
    emb_dim = 768,
    num_heads = 8,
    dropout = 0.1,
    num_layers = 8,
    causal_mask = True
)

In [12]:
total = sum([p.numel() for p in model.parameters()])
print(f"number of parameter : {total}")

number of parameter : 174514513


In [13]:
for name,p in model.named_parameters():
    print(f"{name:<50s} {p.numel()}")
print("=======================================================================================")
print(f"{"SUM TOTAL":<50s} {total}")

token_embedding.weight                             38597376
position_embedding.weight                          786432
transformer_stack.0.layernorm1.weight              768
transformer_stack.0.layernorm1.bias                768
transformer_stack.0.attention.QKV_param.weight     1769472
transformer_stack.0.attention.QKV_param.bias       2304
transformer_stack.0.attention.out_proj.weight      589824
transformer_stack.0.attention.out_proj.bias        768
transformer_stack.0.layernorm2.weight              768
transformer_stack.0.layernorm2.bias                768
transformer_stack.0.ffn.0.weight                   2359296
transformer_stack.0.ffn.0.bias                     3072
transformer_stack.0.ffn.2.weight                   2359296
transformer_stack.0.ffn.2.bias                     768
transformer_stack.1.layernorm1.weight              768
transformer_stack.1.layernorm1.bias                768
transformer_stack.1.attention.QKV_param.weight     1769472
transformer_stack.1.attention.QKV_pa